# Lähde-ekstraktion pohja

**MIKSI tämä notebook on olemassa:**
Yksi tutkimusartikkeli = yksi notebook = yksi `data/interim/{first_author}_{year}.csv` -tiedosto.
Manuaalisesti syötetyt rivit tarkistetaan skeemavalidoinnilla ja SMILES-kanonisoinnilla
ennen tallennusta. Tämä takaa, että jokainen lähde lisätään korpukseen samalla tavalla.

**Käyttöohje:**
1. Kopioi tämä tiedosto nimelle `{first_author}_{year}.ipynb` (esim. `lobmann_2013.ipynb`).
2. Täytä bibliografiset tiedot ja DOI.
3. Syötä rivit `entries`-listaan dictinä per pari.
4. Aja kaikki solut. Jos validointi epäonnistuu, korjaa rivit ja aja uudelleen.
5. Tarkista, että `data/interim/{first_author}_{year}.csv` syntyi oikein.
6. Aja `merge.merge_interim(...)` tai päivitä master-CSV erikseen H1:n päätyttyä.

## (a) Bibliografiset tiedot

In [ ]:
# Täytä tämän lähteen bibliografia. Nämä menevät jokaiseen riviin source_*-sarakkeisiin.
SOURCE = {
    "source_doi": "10.1021/mp2002973",
    "source_first_author": "Lobmann",
    "source_year": 2011,
    "source_table_or_figure": "Table 1",
    "extraction_date": "2026-05-04",
    "extracted_by": "OK",
}

FIRST_AUTHOR_SLUG = SOURCE["source_first_author"].lower()
OUTPUT_FILENAME = f"{FIRST_AUTHOR_SLUG}_{SOURCE['source_year']}.csv"


## (b) Manuaalinen datan syöttö

Yksi `dict` per (drug_A, drug_B) -pari. Jätä puuttuvat arvot `None`:ksi (älä keksi).
Sarakkeet voi tarkistaa `configs/corpus_schema.yaml`:sta.

In [ ]:
# Lähde: Löbmann et al. 2011, "Coamorphous Drug Systems..." Mol. Pharm. 8, 1919-1928.
# DOI: 10.1021/mp2002973. Tg & Tm: Table 1 + Section 2.2.1; stabiilisuus: Section 3.2 + Fig 6.
#
# Säilytysolot: kuiva P2O5-desikaattori 277.15 K (4 °C) ja 298.15 K (25 °C).
# RH ≈ 0 %. Nämä EIVÄT vastaa Baird-Taylor-kriteerin kiihdytettyä (40 °C/75 % RH) eivätkä
# tavanomaista (25 °C/60 % RH). Siksi label_confidence='low' ja gfa_class=2 on käsin
# asetettu (21 vrk:n havaintoikkuna sopii Class II:n alueelle, ei Class I < 7 vrk:n eikä
# Class III ≥ 180 vrk:n alueelle).
#
# Sensurointi (induction_time_censored) merkitty käyttäjän ohjeistuksen mukaan
# yhdenmukaisesti False kaikille; vakaiden rivien (1:1 molemmilla T:llä, 1:2 @ 4 °C)
# osalta tieteellisesti tarkempi olisi censored=True. Notes-kenttä erottaa havainnon
# (kiteytyi vs. pysyi amorfisena), jotta tieto ei katoa myöhempää uudelleenarviointia varten.

# SMILES: PubChem-kanoniset (kanonisoidaan vielä RDKit:lla cell-10:ssä).
NAP_SMILES = "COc1ccc2cc(C(C)C(=O)O)ccc2c1"  # racemaatti; paperi ei eritele enantiomeeriä
IND_SMILES = "CC1=C(C2=CC(=CC=C2N1C(=O)C3=CC=C(C=C3)Cl)OC)CC(=O)O"  # γ-polymorfi
NAP_CAS = "22204-53-1"
IND_CAS = "53-86-1"

# Tg-arvot (K) Table 1; riippumattomia säilytysoloista.
TG_2_1 = (292.04, 0.36)  # 2:1 NAP:IND, NAP-rikas
TG_1_1 = (298.45, 0.24)  # 1:1 NAP:IND, ekvimolaarinen
TG_1_2 = (305.16, 0.22)  # 1:2 NAP:IND, IND-rikas

# Pure component sulamispisteet (Section 2.2.1).
TM_NAP_K = 431.25
TM_IND_K = 435.15


def _row(entry_id, x_nap, x_ind, T_C, tg_pair, outcome_note):
    """Apuri yhden rivin rakentamiseen - vähentää copy-pastea."""
    tg_K, tg_sd = tg_pair
    return {
        "entry_id": entry_id,
        "drug_A_name": "Naproxen",
        "drug_B_name": "Indomethacin",
        "drug_A_role": "api",
        "drug_B_role": "api",
        "drug_A_smiles_original": NAP_SMILES,
        "drug_B_smiles_original": IND_SMILES,
        "drug_A_cas": NAP_CAS,
        "drug_B_cas": IND_CAS,
        "mole_fraction_A": x_nap,
        "mole_fraction_B": x_ind,
        "ratio_reported_as": "mole_fraction",
        "process_method": "quench_cooling",
        "process_details": "Melt at 441.15 K for 5 min, then quench with liquid N2.",
        "gfa_class": 2,
        "gfa_class_label": "Class II",
        "label_confidence": "low",
        "induction_time_days": 21.0,
        "induction_time_censored": False,
        "storage_T_C": T_C,
        "storage_RH_percent": 0.0,
        "Tg_K": tg_K,
        "Tg_uncertainty_K": tg_sd,
        "Tg_heating_rate_K_min": 10.0,
        "Tm_A_K": TM_NAP_K,
        "Tm_B_K": TM_IND_K,
        "pxrd_amorphous": True,  # kaikki kolme seosta amorfisia päivänä 0 (Fig 3)
        "detection_methods": "PXRD,DSC,FTIR",
        "notes": (
            "Kuiva P2O5-desikaattori (RH approx 0 %); olot eivat vastaa "
            "Baird-Taylor-kriteerin kiihdytettya (40/75) eivatka tavanomaista (25/60). "
            "XRPD-tarkistus paivina 0/7/14/21 (havaintoaikavali 7 vrk). " + outcome_note
        ),
    }


entries = [
    # Rivit 1-2: 2:1 NAP:IND (NAP-rikas) - ylimäärä-NAP kiteytyy molemmilla T:llä
    _row("Table1_NAP2-IND1_4C",  0.667, 0.333,  4.0, TG_2_1,
         "Paivana 21 ylimaara-NAP kiteytynyt (Fig 6, piikki 19.1 deg)."),
    _row("Table1_NAP2-IND1_25C", 0.667, 0.333, 25.0, TG_2_1,
         "Paivana 21 ylimaara-NAP kiteytynyt (Fig 6, piikit 19.0, 22.5, 27.4 deg)."),

    # Rivit 3-4: 1:1 NAP:IND - heterodimeeri, vakain pari, pysyy amorfisena
    _row("Table1_NAP1-IND1_4C",  0.500, 0.500,  4.0, TG_1_1,
         "Paivana 21 amorfinen halo - ei kiteytymista havaittu."),
    _row("Table1_NAP1-IND1_25C", 0.500, 0.500, 25.0, TG_1_1,
         "Paivana 21 amorfinen halo - ei kiteytymista havaittu."),

    # Rivit 5-6: 1:2 NAP:IND (IND-rikas) - vakaa @ 4 °C, ylimäärä-IND kiteytyy @ 25 °C
    _row("Table1_NAP1-IND2_4C",  0.333, 0.667,  4.0, TG_1_2,
         "Paivana 21 amorfinen halo - ei kiteytymista havaittu."),
    _row("Table1_NAP1-IND2_25C", 0.333, 0.667, 25.0, TG_1_2,
         "Paivana 21 ylimaara-gamma-IND kiteytynyt (Fig 6, piikit 11.8, 17.2, 22.0, 26.7 deg)."),
]


## (c) DataFrame-konstruktio

In [ ]:
import logging
from pathlib import Path

import pandas as pd

from coamorphous.corpus.canonicalize import canonical_smiles, inchikey_from_smiles, safe_canonical
from coamorphous.corpus.schema import build_schema, column_order, load_schema_yaml

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "configs" / "corpus_schema.yaml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Projektin juurta ei löydy (configs/corpus_schema.yaml puuttuu).")
    PROJECT_ROOT = PROJECT_ROOT.parent

SCHEMA_YAML = PROJECT_ROOT / "configs" / "corpus_schema.yaml"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

spec = load_schema_yaml(SCHEMA_YAML)
cols = column_order(spec)

# Lisätään SOURCE-tiedot ja pair_id jokaiselle riville.
rows = []
for i, e in enumerate(entries, start=1):
    row = {**SOURCE, **e}
    row["pair_id"] = f"{FIRST_AUTHOR_SLUG}{SOURCE['source_year']}_{i:02d}"
    rows.append(row)

df = pd.DataFrame(rows, columns=cols)
df.head()

## (d) Skeemavalidointi

In [ ]:
# Tässä vaiheessa Pandera tarkistaa tyypit ja enumit. Jos tämä epäonnistuu,
# korjaa entries-lista yllä ja aja solut uudelleen.
schema = build_schema(SCHEMA_YAML)

if df.empty:
    print("Ei riviä syötetty - skipataan validointi.")
else:
    df = schema.validate(df, lazy=True)
    print(f"OK: {len(df)} riviä läpäisi skeemavalidoinnin.")

## (e) SMILES-kanonisointi ja InChIKey-haku

In [ ]:
# MIKSI tässä: original-SMILES säilytetään auditointia varten,
# kanoninen muoto käytetään duplikaattitarkistuksiin ja deskriptorilaskentaan.
if not df.empty:
    df["drug_A_smiles_canonical"] = df["drug_A_smiles_original"].apply(safe_canonical)
    df["drug_B_smiles_canonical"] = df["drug_B_smiles_original"].apply(safe_canonical)
    df["drug_A_inchikey"] = df["drug_A_smiles_canonical"].apply(
        lambda s: inchikey_from_smiles(s) if s else None
    )
    df["drug_B_inchikey"] = df["drug_B_smiles_canonical"].apply(
        lambda s: inchikey_from_smiles(s) if s else None
    )
    # Validoidaan uudelleen, koska sarakkeet muuttuivat.
    df = schema.validate(df, lazy=True)
    df[["drug_A_name", "drug_A_smiles_canonical", "drug_A_inchikey"]].head()

## (f) Tallennus interim-tiedostoon

In [ ]:
output_path = INTERIM_DIR / OUTPUT_FILENAME
df.to_csv(output_path, index=False)
print(f"Tallennettu {len(df)} riviä -> {output_path}")